# PapyrusLab E02 — run `infer-46527-seed42` (segmento `pherc0814-46527`)

Notebook generato da `scripts/build_e02_notebooks.py` (non modificare a mano). Piano congelato: `docs/plans/2026-09-06-e02-costruire-il-metro.md`.
Ogni controllo di arresto del piano è un'asserzione: se fallisce, il run si ferma e il log dice dove.

- Output persistiti: `/kaggle/working/e02/out` e `/kaggle/working/e02/logs`
- File pesanti (codice, checkpoint, label, input, cache), non persistiti: `/tmp/e02`


In [ ]:
MODE = "infer-46527-seed42"
KIND = "infer"            # prep | infer
SEG = "pherc0814-46527"              # nome della label, es. pherc0814-46527
SHORT = "46527"
SEED = 42              # None nel prep
SETS = "held,train"            # insiemi di pixel misurati: 'held,train' oppure 'train' (segmento di verifica sigillato)
WORK = "/kaggle/working/e02"
HEAVY = "/tmp/e02"
SRC_URL = "https://vesuvius-challenge-open-data.s3.amazonaws.com/PHerc0814/segments/20260226000000-46527_2um_try2/surface-volumes/2.399um-0.22m-78keV-volume-20260309142202.zarr"      # volume 2,4 um sorgente (solo prep)
LABEL_SHAPE = [21, 2130, 3455]
TORCH_EXPECTED = "2.10.0+cu128"
LABEL_TREE_SHA256 = "5659236870d7d0408e330f05f6275bd821fc1d7bdea8c9c8f072dfd4ae8b54f0"
LABEL_FILES, LABEL_BYTES = 1386, 118869
INPUT_TREE_SHA256 = "bc7423431221bf24b247a8ba80d264b0306f816c52b4ecc0d08115a82305ac52"     # atteso solo nei run infer
print("MODE", MODE, "SEG", SEG, "SEED", SEED, "SETS", SETS, "torch atteso", TORCH_EXPECTED)


In [ ]:
# Guardia iniziale (run GPU): input poolato e label devono essere montati e con hash corretto PRIMA di qualunque
# installazione (piano §4 passo 5, revisione R1 finding 6). Nessun pooling in sessione GPU.
import glob, hashlib, os, json

def tree_sha256(root):
    per_file = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            per_file[os.path.relpath(p, root).replace(os.sep, "/")] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    h = hashlib.sha256()
    for rel, digest in sorted(per_file.items()):
        h.update(f"{rel}\n{digest}\n".encode())
    return h.hexdigest(), len(per_file)

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count("/") - 2
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + "/", "(", len(files), "file )")
hits = glob.glob(f"/kaggle/input/**/{SEG}_pooled.zarr/0/.zarray", recursive=True)
assert hits, f"STOP: input poolato {SEG}_pooled.zarr non montato sotto /kaggle/input: ripararlo con il run prep (CPU), mai qui"
INPUT_ZARR = os.path.dirname(os.path.dirname(hits[0]))
tsha, nfiles = tree_sha256(INPUT_ZARR)
print("input:", INPUT_ZARR, "file", nfiles, "tree_sha256", tsha)
assert tsha == INPUT_TREE_SHA256, f"STOP: tree_sha256 dell'input montato ({tsha}) diverso da quello congelato ({INPUT_TREE_SHA256})"
lab_hits = glob.glob(f"/kaggle/input/**/{SEG}/{SEG}_inklabels.zarr/0/.zarray", recursive=True)
assert lab_hits, f"STOP: label {SEG} non montata sotto /kaggle/input (dataset delle label assente)"
print("label montata:", os.path.dirname(os.path.dirname(os.path.dirname(lab_hits[0]))))
open(f"/kaggle/working/e02_guard.json", "w").write(json.dumps({"input_zarr": INPUT_ZARR, "input_tree_sha256": tsha, "input_files": nfiles}))


In [ ]:
%%bash
# Passo 1 — radice misurabile, cache e temporanei dirottati, guardia dei 15 GB
set -e
mkdir -p /kaggle/working/e02/out /kaggle/working/e02/logs /tmp/e02/tmp /tmp/e02/cache/pip /tmp/e02/cache/hf /tmp/e02/checkpoints /tmp/e02/labels /tmp/e02/input
cat > /kaggle/working/e02/env.sh <<'EOF'
export WORK=/kaggle/working/e02
export HEAVY=/tmp/e02
export TMPDIR=$HEAVY/tmp PIP_CACHE_DIR=$HEAVY/cache/pip HF_HOME=$HEAVY/cache/hf
export LIMIT_BYTES=$((15*1024*1024*1024))
disk_check () {
  local used
  used=$(( $(du -sb "$WORK" | cut -f1) + $(du -sb "$HEAVY" | cut -f1) ))
  echo "spazio_byte=$used ($1)" | tee -a "$WORK/logs/disk_check.log"
  if [ "$used" -gt "$LIMIT_BYTES" ]; then echo "STOP: superati 15 GB ($1)" | tee -a "$WORK/logs/disk_check.log"; exit 1; fi
}
EOF
source /kaggle/working/e02/env.sh
echo "MODE=infer-46527-seed42 SEG=pherc0814-46527 SEED=42 start=$(date -u +%FT%TZ)" > $WORK/logs/run_info.txt
disk_check "inizio"
df -h /kaggle/working /tmp | tail -2


In [ ]:
# Passo 1 (segue) — versioni dell'ambiente e rete verso le sorgenti
import sys, platform, json, urllib.request, torch
env = {"python": sys.version.split()[0], "platform": platform.platform(),
       "torch": torch.__version__, "cuda_available": torch.cuda.is_available(),
       "cuda_device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0}
print(json.dumps(env, indent=1))
json.dump(env, open(f"{WORK}/logs/env_before_install.json", "w"), indent=1)
assert env["torch"] == TORCH_EXPECTED, f"STOP: PyTorch inatteso {env['torch']} (atteso {TORCH_EXPECTED}): Kaggle ha cambiato immagine, aggiornare il piano"
if KIND == "prep":
    assert not env["cuda_available"], "STOP: il run prep deve girare senza acceleratore"
else:
    assert env["cuda_available"], "STOP: run GPU senza CUDA disponibile"
urls = ["https://huggingface.co/api/models/scrollprize/ink_9um", "https://huggingface.co/api/buckets/scrollprize/datasets"]
if KIND == "prep":
    urls.insert(0, SRC_URL + "/2/.zarray")
for url in urls:
    with urllib.request.urlopen(url, timeout=30) as r:
        print(r.status, url[:90]); assert r.status == 200, f"STOP: rete non raggiunge {url}"


In [ ]:
%%bash
# Passo 2 — checkout parziale di villa al commit congelato
set -e
source /kaggle/working/e02/env.sh
cd $HEAVY
[ -d villa/.git ] || git clone -q --filter=blob:none --no-checkout https://github.com/ScrollPrize/villa.git
cd villa
git sparse-checkout init --cone >/dev/null
git sparse-checkout set ink-detection vesuvius >/dev/null
git checkout -q 3ea17f54a9b3d5fd1aaf73e1d2c8386dbaa9f30e
HEAD=$(git rev-parse HEAD); echo "villa HEAD=$HEAD" | tee $WORK/logs/villa_commit.txt
[ "$HEAD" = "3ea17f54a9b3d5fd1aaf73e1d2c8386dbaa9f30e" ] || { echo "STOP: commit villa diverso"; exit 1; }
ls -l ink-detection/koine_machines/inference/infer.py ink-detection/scripts/prepare_9um_isotropic_input.py vesuvius/pyproject.toml ink-detection/uv.lock
sha256sum ink-detection/scripts/prepare_9um_isotropic_input.py | tee $WORK/logs/prepare_script_sha256.txt
disk_check "dopo checkout"


In [ ]:
# Passo 3 — installazione sul Python di sistema, tutto con --no-deps, PyTorch intatto (cella di E00, invariata)
import subprocess, re, json, sys
VILLA = f"{HEAVY}/villa"
PY = sys.executable

def sh(args):
    return subprocess.run(args, capture_output=True, text=True)

def pip(*args):
    return sh([PY, "-m", "pip", "install", "--no-deps", *args])

def torch_version():
    return sh([PY, "-c", "import torch; print(torch.__version__)"]).stdout.strip()

torch_before = torch_version()
for pkg in ["vesuvius", "ink-detection"]:
    r = pip("-e", f"{VILLA}/{pkg}")
    assert r.returncode == 0, "STOP: pip install fallita\n" + r.stderr[-3000:]
r = pip("zarr==2.18.7", "numcodecs==0.15.1")   # il codice usa l'API Zarr v2
assert r.returncode == 0, "STOP: pip install zarr/numcodecs fallita\n" + r.stderr[-3000:]

lock = open(f"{VILLA}/ink-detection/uv.lock", encoding="utf-8").read()
def locked_version(dist):
    m = re.search(r'\[\[package\]\]\nname = "' + re.escape(dist.lower()) + r'"\nversion = "([^"]+)"', lock)
    return m.group(1) if m else None

ALIAS = {"cv2": "opencv-contrib-python-headless", "PIL": "pillow", "yaml": "pyyaml", "nrrd": "pynrrd",
         "cc3d": "connected-components-3d", "skimage": "scikit-image", "sklearn": "scikit-learn"}
added, numpy_downgraded = [], False

def ensure_import(modname):
    global numpy_downgraded
    for attempt in range(12):
        r = sh([PY, "-c", f"import {modname}"])
        if r.returncode == 0:
            return
        err = r.stderr
        m = re.search(r"No module named '([^'.]+)", err)
        if m:
            mod = m.group(1)
            assert re.fullmatch(r"[A-Za-z0-9_]+", mod), f"STOP: nome di modulo inatteso {mod!r}"
            cands = ([ALIAS[mod]] if mod in ALIAS else []) + [mod, mod.replace("_", "-")]
            dist = next((c for c in cands if locked_version(c)), None)
            assert dist, f"STOP: modulo mancante '{mod}' non presente in uv.lock:\n" + err[-2000:]
            ver = locked_version(dist)
            r2 = pip(f"{dist}=={ver}")
            assert r2.returncode == 0, f"STOP: pip install {dist}=={ver} fallita\n" + r2.stderr[-3000:]
            added.append({"module": mod, "dist": dist, "version": ver}); print("aggiunto", dist, ver, "per", modname)
        elif "numpy" in err.lower() and not numpy_downgraded:
            r2 = pip("numpy<=2.2"); assert r2.returncode == 0, r2.stderr[-3000:]
            numpy_downgraded = True; added.append({"module": "numpy", "dist": "numpy", "version": "<=2.2 (eccezione piano)"})
        else:
            raise AssertionError(f"STOP: import di {modname} fallito per motivo diverso da modulo mancante:\n" + err[-3000:])
    raise AssertionError(f"STOP: import di {modname} ancora fallito dopo i tentativi ammessi")

for modname in ["koine_machines.inference.infer", "vesuvius", "vesuvius.models.build.build_network_from_config",
                "koine_machines.models.make_model"]:
    ensure_import(modname)
for lazy in ["imagecodecs"]:
    if sh([PY, "-c", f"import {lazy}"]).returncode != 0:
        ver = locked_version(lazy); assert ver, f"STOP: {lazy} non presente in uv.lock"
        r2 = pip(f"{lazy}=={ver}"); assert r2.returncode == 0, f"STOP: pip install {lazy}=={ver} fallita\n" + r2.stderr[-3000:]
        added.append({"module": lazy, "dist": lazy, "version": ver}); print("aggiunto", lazy, ver, "(dipendenza pigra di tifffile)")
assert len(added) <= 10, f"STOP: {len(added)} pacchetti aggiunti, oltre il limite di dieci del piano"

torch_after = torch_version()
assert torch_after == torch_before == TORCH_EXPECTED, f"STOP: PyTorch cambiato da {torch_before} a {torch_after}"
helptxt = sh([PY, "-m", "koine_machines.inference.infer", "--help"]).stdout
for flag in ["--no-compile", "--gpus", "--layer-start"]:
    assert flag in helptxt, f"STOP: opzione {flag} assente nell'entry point"
hf = sh(["hf", "--version"])
if hf.returncode != 0:
    r2 = pip("huggingface_hub"); assert r2.returncode == 0, r2.stderr[-2000:]
    hf = sh(["hf", "--version"]); added.append({"module": "hf", "dist": "huggingface_hub", "version": hf.stdout.strip()})
pipl = [l for l in sh([PY, "-m", "pip", "list"]).stdout.splitlines()
        if re.match(r"(?i)^(torch|torchvision|zarr|numcodecs|numpy|tifffile|timm|scipy|fsspec|s3fs|aiohttp|huggingface.hub|monai|koine.machines|vesuvius|albumentations|einops|opencv|imagecodecs) ", l)]
open(f"{WORK}/logs/pip_versions.txt", "w").write("\n".join(pipl) + "\n")
info = {"torch_before": torch_before, "torch_after": torch_after, "added_packages": added, "hf_version": hf.stdout.strip()}
json.dump(info, open(f"{WORK}/logs/install.json", "w"), indent=1)
print(json.dumps(info, indent=1)); print("\n".join(pipl))


In [ ]:
%%bash
# Passo 4 — checkpoint con verifica esatta di hash e dimensioni (cella di E00)
set -e
source /kaggle/working/e02/env.sh
cd $HEAVY
disk_check "prima dei checkpoint"
hf download scrollprize/ink_9um hybrid_3d2d-seed42/step-075000.pth hybrid_3d2d-seed43/step-075000.pth \
   --revision 7109667e2607db1b90c37c8b09cb876ea7fe7bb1 --local-dir checkpoints/ink_9um > /dev/null
sha256sum checkpoints/ink_9um/hybrid_3d2d-seed42/step-075000.pth checkpoints/ink_9um/hybrid_3d2d-seed43/step-075000.pth | tee $WORK/logs/checkpoints_sha256.txt
stat -c "%s %n" checkpoints/ink_9um/hybrid_3d2d-seed4*/step-075000.pth | tee -a $WORK/logs/checkpoints_sha256.txt
grep -q "^e635558ae6a1a807a7e5ec1e83adfd45bc3c0ac53883ea43f1d4e085d62a9cab  checkpoints/ink_9um/hybrid_3d2d-seed42/step-075000.pth" $WORK/logs/checkpoints_sha256.txt || { echo "STOP: SHA-256 seed42 diverso"; exit 1; }
grep -q "^2aeaa85a35ef28d7bc7bf3e848c4a6a91385e9132710927fdba41133c4ecb28f  checkpoints/ink_9um/hybrid_3d2d-seed43/step-075000.pth" $WORK/logs/checkpoints_sha256.txt || { echo "STOP: SHA-256 seed43 diverso"; exit 1; }
disk_check "dopo i checkpoint"


In [ ]:
# Passo 4 (segue) — label del segmento dal dataset Kaggle privato (ricerca ricorsiva: il mount cambia fra sessioni
# CPU e GPU, lezione di E00), verificata file per file contro manifest.json e per contenuto (tree_sha256);
# ripiego: download diretto dal bucket con 4 thread e attesa crescente (HTTP 429), come E00.
import hashlib, os, shutil, json, glob, re, time, urllib.request, concurrent.futures as cf
PREFIX = f"ink_9um/labels/aligned-scrollprizeorg-21slices/{SEG}/"
API = "https://huggingface.co/api/buckets/scrollprize/datasets/tree/" + PREFIX.rstrip("/")
RESOLVE = "https://huggingface.co/buckets/scrollprize/datasets/resolve/"
DEST = f"{HEAVY}/labels/{SEG}"

def tree_sha256(root):
    per_file = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            per_file[os.path.relpath(p, root).replace(os.sep, "/")] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    h = hashlib.sha256()
    for rel, digest in sorted(per_file.items()):
        h.update(f"{rel}\n{digest}\n".encode())
    return h.hexdigest(), per_file

def check_label_tree(dest, source_desc):
    n = sum(len(fs) for _, _, fs in os.walk(dest)); b = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(dest) for f in fs)
    assert (n, b) == (LABEL_FILES, LABEL_BYTES), f"STOP: label con {n} file / {b} byte, attesi {(LABEL_FILES, LABEL_BYTES)}"
    tsha, _ = tree_sha256(dest)
    assert tsha == LABEL_TREE_SHA256, f"STOP: contenuto della label diverso da quello congelato (tree sha256 {tsha})"
    open(f"{WORK}/logs/label_count.txt", "w").write(f"seg={SEG} file={n} byte={b} tree_sha256={tsha} source={source_desc}\n")
    za = json.load(open(f"{dest}/{SEG}_inklabels.zarr/0/.zarray")); open(f"{WORK}/logs/label_zarray.json", "w").write(json.dumps(za))
    assert za["shape"] == LABEL_SHAPE, f"STOP: forma della label {za['shape']} diversa da {LABEL_SHAPE}"
    print(f"label verificata: file={n} byte={b} tree_sha256={tsha} ({source_desc}) shape={za['shape']}")

roots = glob.glob(f"/kaggle/input/**/{SEG}/{SEG}_inklabels.zarr/0/.zarray", recursive=True)
manifests = [m for m in glob.glob("/kaggle/input/**/manifest.json", recursive=True) if SEG in json.load(open(m)).get("segments", {})]
print("label montata:", roots[:1], "| manifest:", manifests[:1])
if os.path.isdir(DEST):
    shutil.rmtree(DEST)
if roots and manifests:
    src = os.path.dirname(os.path.dirname(os.path.dirname(roots[0])))       # .../<SEG>
    man = json.load(open(manifests[0]))["segments"][SEG]
    files = man["files"]
    assert (len(files), sum(f["size"] for f in files)) == (LABEL_FILES, LABEL_BYTES), "STOP: manifest del dataset diverso dalle costanti congelate"
    missing = [f["path"] for f in files if not (os.path.isfile(os.path.join(src, f["path"][len(PREFIX):])) and os.path.getsize(os.path.join(src, f["path"][len(PREFIX):])) == f["size"])]
    assert not missing, f"STOP: {len(missing)} file della label mancanti o di dimensione diversa, p.es. {missing[:3]}"
    shutil.copytree(src, DEST)
    check_label_tree(DEST, f"kaggle_dataset manifest_sha256={hashlib.sha256(open(manifests[0], 'rb').read()).hexdigest()}")
else:
    assert KIND == "prep", "STOP: nei run GPU la label deve essere montata (nessun ripiego di rete con la GPU allocata)"
    def list_label_files():
        files, url = [], API
        while url:
            req = urllib.request.Request(url, headers={"User-Agent": "papyruslab-e02"})
            with urllib.request.urlopen(req, timeout=60) as r:
                files += [(e["path"], int(e["size"])) for e in json.load(r) if e.get("type") == "file"]
                m = re.search(r'<([^>]+)>;\s*rel="next"', r.headers.get("Link", "") or "")
                url = m.group(1) if m else None
        return files
    def fetch(item):
        path, size = item
        assert path.startswith(PREFIX) and ".." not in path, f"STOP: percorso inatteso {path}"
        out = os.path.join(DEST, path[len(PREFIX):])
        if os.path.exists(out) and os.path.getsize(out) == size:
            return size
        os.makedirs(os.path.dirname(out), exist_ok=True)
        last = None
        for attempt in range(8):
            try:
                req = urllib.request.Request(RESOLVE + path, headers={"User-Agent": "papyruslab-e02"})
                with urllib.request.urlopen(req, timeout=60) as r:
                    data = r.read()
                if len(data) == size:
                    open(out, "wb").write(data); return size
                last = f"dimensione {len(data)} != {size}"
            except Exception as ex:
                last = ex
            time.sleep(min(60, 5 * 2 ** attempt))
        raise RuntimeError(f"STOP: download fallito per {path}: {last}")
    files = list_label_files(); total = sum(s for _, s in files)
    assert (len(files), total) == (LABEL_FILES, LABEL_BYTES), f"STOP: label diversa dalla misura congelata: {len(files)} file, {total} byte"
    t0 = time.time()
    with cf.ThreadPoolExecutor(max_workers=4) as ex:
        got = sum(ex.map(fetch, files))
    print(f"scaricati {got} byte in {time.time() - t0:.0f} s")
    check_label_tree(DEST, "direct_download")


In [ ]:
# Passo 5 (infer) — costruire il modello dal checkpoint su CPU come fa infer.py, in sottoprocesso (cella di E00)
import subprocess, sys, json
code = r'''
import argparse, json, torch
from koine_machines.inference import infer as kinfer
args = argparse.Namespace(checkpoint="__CKPT__", amp_dtype="auto", model_type="auto", metadata_json=None)
cm = kinfer.configure_model(args)
info = {"in_chans": int(cm.in_chans), "amp_dtype": str(cm.amp_dtype),
        "n_params": int(sum(p.numel() for p in cm.model.parameters())),
        "model_class": type(cm.model).__name__, "preprocessing": str(cm.preprocessing)[:200]}
print("MODEL_BUILD_JSON=" + json.dumps(info))
'''.replace("__CKPT__", f"{HEAVY}/checkpoints/ink_9um/hybrid_3d2d-seed{SEED}/step-075000.pth")
r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-1500:])
assert r.returncode == 0, "STOP: costruzione del modello su CPU fallita (vedi stderr sopra)"
info = json.loads(r.stdout.split("MODEL_BUILD_JSON=")[1].splitlines()[0])
json.dump(info, open(f"{WORK}/logs/model_build_cpu.json", "w"), indent=1)
assert info["in_chans"] == 17, "STOP: il checkpoint non dichiara 17 slice in ingresso"
assert "float16" in info["amp_dtype"], f"STOP: AMP dtype inatteso {info['amp_dtype']}"
print("modello costruito su CPU:", info)


In [ ]:
%%bash
# Passo 7 — inferenza seed 42 sull'input poolato montato, una sola T4 (CUDA_VISIBLE_DEVICES=0), timeout 30 minuti
source /kaggle/working/e02/env.sh
cd $HEAVY
INPUT_ZARR=$(python -c "import json; print(json.load(open('/kaggle/working/e02_guard.json'))['input_zarr'])")
echo "input=$INPUT_ZARR" | tee $WORK/logs/input_path.txt
disk_check "prima inferenza seed42"
nvidia-smi --query-gpu=index,name,memory.total,memory.used,driver_version --format=csv > $WORK/logs/nvidia-smi-before-seed42.txt \
  || { echo "STOP: nvidia-smi non disponibile: nessuna GPU assegnata"; exit 1; }
cat $WORK/logs/nvidia-smi-before-seed42.txt
export CUDA_VISIBLE_DEVICES=0
python -c "import torch; assert torch.cuda.is_available(); print(torch.__version__, 'visibili', torch.cuda.device_count(), torch.cuda.get_device_name(0))" | tee -a $WORK/logs/env_gpu.txt \
  || { echo "STOP: PyTorch non vede la GPU"; exit 1; }
nvidia-smi --query-gpu=timestamp,index,memory.used --format=csv,noheader,nounits -l 5 > $WORK/logs/gpu_samples_seed42.csv &
SAMPLER=$!
set -o pipefail
START=$(date +%s)
timeout -s INT -k 30 1800 python -m koine_machines.inference.infer \
  "$INPUT_ZARR" checkpoints/ink_9um/hybrid_3d2d-seed42/step-075000.pth $WORK/out/pherc0814-46527_seed42_step075000.tif \
  --overlap 0.5 --blend-mode hann --no-compile --gpus 0 --batch-size 1 \
  2>&1 | tee $WORK/logs/infer_seed42.log
EXIT=${PIPESTATUS[0]}; END=$(date +%s)
kill $SAMPLER 2>/dev/null; wait $SAMPLER 2>/dev/null
CAUSA=normale; [ "$EXIT" -eq 124 ] && CAUSA=timeout_1800s
echo "exit_code=$EXIT durata_s=$((END-START)) causa=$CAUSA" | tee -a $WORK/logs/infer_seed42.log
nvidia-smi --query-gpu=index,memory.used --format=csv > $WORK/logs/nvidia-smi-after-seed42.txt; cat $WORK/logs/nvidia-smi-after-seed42.txt
disk_check "dopo inferenza seed42"


In [ ]:
# Passo 7 (segue) — controlli sul log: codice di uscita, finestra Z 2-18, canali, GPU singola misurata (cella di E00 adattata)
import re, csv
log = open(f"{WORK}/logs/infer_seed{SEED}.log", encoding="utf-8", errors="replace").read()
m = re.search(r"exit_code=(\d+) durata_s=(\d+) causa=(\S+)", log); assert m, "STOP: riga finale di esito assente nel log"
exit_code, durata, causa = int(m.group(1)), int(m.group(2)), m.group(3)
print("exit_code", exit_code, "durata_s", durata, "causa", causa)
assert exit_code == 0, f"STOP: inferenza terminata con exit_code={exit_code} ({causa})"
expected = "Selected source layer indices=" + str([2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
assert expected in log, "STOP: indici di layer diversi da 2-18 (input a 21 slice)"
assert "in_chans=17" in log, "STOP: in_chans diverso da 17"
assert "Using CUDA device 0 for inference." in log, "STOP: il log non conferma l'uso del solo device 0"
assert re.search(r"Wrote .*%s_seed%d_step075000.tif" % (SEG, SEED), log), "STOP: il log non conferma la scrittura del TIFF"
before = open(f"{WORK}/logs/nvidia-smi-before-seed{SEED}.txt").read()
gpus_before = [l for l in before.splitlines()[1:] if l.strip()]
assert gpus_before and all("T4" in l for l in gpus_before), f"STOP: GPU inattese o assenti prima del run: {gpus_before}"
peak = {}
for row in csv.reader(open(f"{WORK}/logs/gpu_samples_seed{SEED}.csv")):
    if len(row) >= 3 and row[1].strip().isdigit():
        idx, mem = int(row[1]), int(row[2]); peak[idx] = max(peak.get(idx, 0), mem)
print("campioni memoria (MiB, picco per GPU):", peak, "| GPU allocate:", len(gpus_before))
assert peak.get(0, 0) > 200, "STOP: nessun uso misurato della GPU 0 durante l'inferenza"
for idx in peak:
    if idx != 0:
        assert peak[idx] < 100, f"STOP: la GPU {idx} ha usato {peak[idx]} MiB: il vincolo di una sola GPU non e' rispettato"
open(f"{WORK}/logs/gpu_peak_seed{SEED}.json", "w").write(str(peak))


In [ ]:
# Passo 8 — metriche minime con lo script congelato scripts/e02_metrics.py (inlineato dal generatore: il repository e' privato)
import json, hashlib, os, sys, importlib, base64
METRICS_SOURCE = base64.b64decode("...").decode("utf-8")
open(f"{WORK}/e02_metrics.py", "w", encoding="utf-8", newline="\n").write(METRICS_SOURCE)
open(f"{WORK}/logs/e02_metrics_sha256.txt", "w").write(hashlib.sha256(METRICS_SOURCE.encode("utf-8")).hexdigest() + "\n")
sys.path.insert(0, WORK); m = importlib.import_module("e02_metrics")
from pathlib import Path
tif = Path(f"{WORK}/out/{SEG}_seed{SEED}_step075000.tif")
sets = tuple(s for s in SETS.split(",") if s)
rep = m.build_report(tif, Path(f"{HEAVY}/labels/{SEG}"), sets=sets, threshold=None, edges=(0, 64, 128, 256), patch=128)
assert rep["shape_ok"], f"STOP: TIFF {rep['shape']} {rep['dtype']} diverso dalla label"
assert "held" not in rep["sets"] or SETS != "train", "STOP: insieme held calcolato su un segmento sigillato"
train = rep["sets"]["train"]
gate_A = "superato"     # identita': commit, hash checkpoint, label, indici 2-18, exit 0, forma: tutti asseriti nelle celle precedenti
ok_orient = train["orientation"]["orientamento_ok"]
disjoint = rep["disjoint_check"]["n_px_held_and_train"] if rep["disjoint_check"] else None
gate_B = "superato" if (ok_orient is True and disjoint == 0) else ("non_valutabile" if ok_orient is None else "fallito")
rep.update({"gate_A": gate_A, "gate_B": gate_B, "seed": SEED, "segment_label": SEG})
json.dump(rep, open(f"{WORK}/out/metrics_{SEG}_seed{SEED}.json", "w"), indent=1, sort_keys=True)
print("AUROC train", train["auroc"], "| bestF1 train", train["best_f1"], "| orientamento", train["orientation"], "| held&train", disjoint)
if "held" in rep["sets"]:
    h = rep["sets"]["held"]; print("AUROC held", h["auroc"], "| bestF1 held", h["best_f1"], "| within patch", h["within_patch"])
print("GATE A:", gate_A, "| GATE B:", gate_B)


In [ ]:
%%bash
# Persistenza — hash di tutto cio' che viene conservato, stato finale
set -e
source /kaggle/working/e02/env.sh
cp /kaggle/working/e02/env.sh $WORK/logs/env.sh.txt
[ -f /kaggle/working/e02_guard.json ] && cp /kaggle/working/e02_guard.json $WORK/logs/guard.json
echo "end=$(date -u +%FT%TZ)" >> $WORK/logs/run_info.txt
disk_check "finale"
cd $WORK && find out logs -type f ! -name SHA256SUMS -print0 | sort -z | xargs -0 sha256sum > out/SHA256SUMS
cat out/SHA256SUMS
echo "persistito: $(du -sh $WORK | cut -f1)"


In [ ]:
# Verdetto del run: dopo la persistenza, un gate non superato rende il run 'error'
import json
res = json.load(open(f"{WORK}/out/metrics_{SEG}_seed{SEED}.json"))
print("gate_A =", res["gate_A"], "| gate_B =", res["gate_B"], "| AUROC train =", res["sets"]["train"]["auroc"])
assert res["gate_A"] == "superato" and res["gate_B"] == "superato", f"E02 run {MODE} NON superato: gate_A={res['gate_A']} gate_B={res['gate_B']} (metriche e log persistiti)"
